## 1장 1강 : LLM 애플리케이션의 입력과 출력 구조

### 의존 패키지 설치
```
uv add ipykernel langchain langchain-core langchain-ollama python-dotenv
```

### 3. LCEL 파이프라인 실습

#### 3.1 환경 변수 로드 및 패키지 불러오기

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate


#### 3.2 ChatPromptTemplate으로 입력 템플릿 조립하기

시스템 역할과 사용자 질문 변수({user_question})를 담은 템플릿 생성<br>
system: "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."<br>
user: "{user_question}"

In [3]:
# system: 역할, 상황, 제한 조건, 예시 -> 시스템 메세지, SystemMessage(..)
# user: 사용자의 질의 : HumanMessage(..)
# asistant: AI의 답변 : AIMessage(..)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."),
    ("user", "{user_question}")
])

템플릿에 텍스트용 질문 데이터를 주입하여 결과를 확인

user_question: "프로그래밍에서 '변수'가 무엇인가요?"

In [4]:
sample_prompt = prompt_template.invoke({
    "user_question": "프로그래밍에서 '변수'가 무엇인가요?"
})

sample_prompt

ChatPromptValue(messages=[SystemMessage(content='당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content="프로그래밍에서 '변수'가 무엇인가요?", additional_kwargs={}, response_metadata={})])

#### 3.3 ChatOllama로 mistral 모델 호출하기

ChatOllama 모델 인스턴스 생성

In [5]:
model = ChatOllama(
    model="mistral",
    #base_url="http://localhost:11434" #  Ollama 서버 주소
)

model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, model='mistral')

앞서 만든 sample_prompt를 모델에 직접 전달하여 실행

In [6]:
response = model.invoke(sample_prompt)

response

AIMessage(content=' 변수는 컴퓨터 프로그램에서 값(데이터)을 저장하고 조작하는 공간입니다. 예를 들어, 가방은 우리 일상에서 변수와 같습니다. 그녀가 내용물(값)을 가질 수 있고, 내용물을 바꾸고, 추적하는 것을 통해 우리는 가방 안의 데이터를 조작할 수 있습니다.', additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2026-09-09T05:45:57.167126Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11781564334, 'load_duration': 4297463917, 'prompt_eval_count': 106, 'prompt_eval_duration': 550941000, 'eval_count': 147, 'eval_duration': 6930631000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'}, id='lc_run--01a084b3-68a3-7490-ad39-e1ff06640bf6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 147, 'total_tokens': 253})

#### 3.4 StrOutputParser로 순수 텍스트만 추출하기

문자열 출력 파서 생성

In [8]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

raw_response 객체에서 순수 텍스트만 추출

In [9]:
text = parser.invoke(response)

text

' 변수는 컴퓨터 프로그램에서 값(데이터)을 저장하고 조작하는 공간입니다. 예를 들어, 가방은 우리 일상에서 변수와 같습니다. 그녀가 내용물(값)을 가질 수 있고, 내용물을 바꾸고, 추적하는 것을 통해 우리는 가방 안의 데이터를 조작할 수 있습니다.'

#### 3.5 LCEL 파이프(|) 연산자로 완전한 체인 결합 및 실행하기

파이프(|) 연산자를 사용해 3개 컴포넌트를 하나의 파이프라인으로 연결

In [10]:
chain = prompt_template | model | parser

새로운 질문으로 파이프라인 전체 실행

In [12]:
res = chain.invoke({
    "user_question": "클래스(Class)에 대해서 알기 쉬운 비유를 통해 설명하세요"
})

res

' 클래스(Class)는 실제 세계에서 유형(Species)과 같습니다. 예를 들어, 고양이와 개는 유형이라고 보시면 좋습니다. 고양이 유형에는 모든 고양이들이 공통된 특징이 있고, 개 유형에는 모든 개들이 공통된 특징이 있습니다. 이렇게 공통된 특징을 정의하고 이를 클래스(Class)로 만들어 공유할 수 있습니다. 클래스로 만든 개 또는 고양이를 객체(Instance)라고 부르며, 이 객체들은 각각 고유한 속성(property)이나 행동(method)을 가질 수 있습니다. 이렇게 객체는 클래스로부터 공통된 특징을 상속받고, 각자의 고유한 속성을 가지고 있습니다.'

### 4. 프롬프트 엔지니어링
#### 4.1 프롬프트 엔지니어링의 3대 핵심 역할
- 목표 명확화
- 제약 조건 부여
- 출력 규격 표준화

#### 4.2 AI 성능 확장 4단계 비교
- Prompt Engineering
- RAG (검색 증강 생성)
- Fine-tuning (미세 조정)
- AI Agent (에이전트)